# ML Risk Simulator Experiment

Notebook ini digunakan untuk menguji model **post-fulfillment low-rating risk simulator** pada project FP MCI Customer Experience.

Tujuan notebook:
- load artefak model dari folder `models/`;
- melihat daftar fitur input yang dibutuhkan model;
- menjalankan prediksi pada sample input;
- membuat eksperimen skenario manual, misalnya order on-time vs late, single seller vs multi seller, dan order bernilai tinggi;
- membaca output `low_rating_risk`, `low_rating_risk_percentage`, dan `risk_level`.

Catatan penting: model ini bukan causal model dan bukan pre-order prediction system. Model ini hanya risk simulator berbasis sinyal post-fulfillment/delivery.

## 1. Setup Path dan Import

Jalankan notebook dari root repository atau dari folder `notebooks/`. Cell ini akan mencari root repo secara otomatis.

In [2]:
# %pip install scikit-learn==1.6.1 lightgbm joblib pandas numpy

In [3]:
from pathlib import Path
import sys
import json
import pandas as pd

current = Path.cwd().resolve()
repo_root = current if (current / "models").exists() else current.parent
models_dir = repo_root / "models"

sys.path.insert(0, str(models_dir))

print("Repo root:", repo_root)
print("Models dir:", models_dir)
print("Model files:")
for path in sorted(models_dir.glob("*")):
    print("-", path.name)

Repo root: C:\Kuliah ITS Farhan\Semester 4\Admin_Lab\MCI\FP_MCI_Farhan\farhan_fp_mci_customer_experience
Models dir: C:\Kuliah ITS Farhan\Semester 4\Admin_Lab\MCI\FP_MCI_Farhan\farhan_fp_mci_customer_experience\models
Model files:
- __pycache__
- calibrated_lgbm_low_rating.pkl
- feature_schema.json
- predict_risk.py
- preprocessor.pkl
- README.md
- sample_inputs.json


## 2. Load Model Artifacts

Artefak yang dipakai:
- `calibrated_lgbm_low_rating.pkl`
- `preprocessor.pkl`
- `feature_schema.json`
- `sample_inputs.json`

Jika ada error saat load pickle, biasanya penyebabnya adalah versi dependency tidak sama dengan saat model dibuat. Notebook ML final menggunakan scikit-learn sekitar `1.6.1`.

In [4]:
from predict_risk import load_artifacts, predict_risk, predict_batch

model, preprocessor, schema = load_artifacts()

with open(models_dir / "sample_inputs.json", encoding="utf-8") as f:
    samples = json.load(f)

print("Loaded model:", schema.get("model_name"))
print("Model framing:", schema.get("framing"))
print("Final threshold:", schema.get("final_threshold"))
print("Number of input features:", len(schema["feature_columns"]))

Loaded model: Calibrated LightGBM (isotonic)
Model framing: Post-delivery low-rating risk estimator
Final threshold: 0.3
Number of input features: 24


## 3. Lihat Schema Fitur Input

Cell ini membantu memahami input apa saja yang harus diisi jika ingin mencoba skenario manual.

In [5]:
feature_table = pd.DataFrame({
    "feature": schema["feature_columns"],
    "type": [
        "numeric" if col in schema["numeric_features"] else "categorical"
        for col in schema["feature_columns"]
    ]
})

feature_table

,feature,type
0,approval_delay_days,numeric
1,carrier_delay_days,numeric
2,last_mile_days,numeric
3,estimated_delivery_days,numeric
4,actual_delivery_days,numeric
5,delivery_delay_vs_estimate,numeric
6,is_late,numeric
7,item_count,numeric
8,seller_count,numeric
9,total_price,numeric


## 4. Prediksi Sample Input Bawaan

`sample_inputs.json` berisi contoh skenario Low, Medium, dan High. Kolom dengan prefix `_example_` hanya label referensi, bukan fitur model.

In [6]:
def clean_sample(sample: dict) -> dict:
    return {k: v for k, v in sample.items() if not k.startswith("_")}

sample_rows = []
for idx, sample in enumerate(samples, start=1):
    input_dict = clean_sample(sample)
    pred = predict_risk(input_dict, model, preprocessor, schema)
    sample_rows.append({
        "sample_id": idx,
        "expected_reference_level": sample.get("_example_risk_level"),
        "expected_reference_risk": sample.get("_example_low_rating_risk"),
        **pred,
        "is_late": input_dict["is_late"],
        "delivery_delay_vs_estimate": input_dict["delivery_delay_vs_estimate"],
        "actual_delivery_days": input_dict["actual_delivery_days"],
        "customer_state": input_dict["customer_state"],
        "seller_state": input_dict["seller_state"],
        "main_product_category": input_dict["main_product_category"]
    })

pd.DataFrame(sample_rows)

,sample_id,expected_reference_level,expected_reference_risk,low_rating_risk,low_rating_risk_percentage,risk_level,is_late,delivery_delay_vs_estimate,actual_delivery_days,customer_state,seller_state,main_product_category
0,1,Low,0.0610,0.0610,6.10,Low,0,-7.107488,8.436574,SP,SP,housewares
1,2,Medium,0.3119,0.3119,31.19,Medium,0,-25.269005,18.400799,SC,SP,baby
2,3,High,0.7832,0.7832,78.32,High,1,11.933171,21.327963,SP,SP,health_beauty


## 5. Eksperimen Skenario Manual

Bagian ini membuat beberapa skenario dari satu order dasar. Tujuannya untuk melihat bagaimana risiko berubah saat sinyal fulfillment/order complexity diubah.

Interpretasi harus hati-hati: perubahan skor adalah output model berdasarkan pola historis, bukan bukti kausal bahwa satu fitur menyebabkan rating rendah.

In [7]:
base_order = clean_sample(samples[0])

scenario_on_time = base_order.copy()
scenario_on_time.update({
    "scenario": "A - on-time, local, simple order",
    "is_late": 0,
    "delivery_delay_vs_estimate": -7.0,
    "actual_delivery_days": 8.0,
    "carrier_delay_days": 2.3,
    "last_mile_days": 6.0,
    "seller_count": 1,
    "item_count": 1,
    "customer_state": "SP",
    "seller_state": "SP",
    "same_customer_seller_state": 1
})

scenario_late = base_order.copy()
scenario_late.update({
    "scenario": "B - late delivery, long carrier delay",
    "is_late": 1,
    "delivery_delay_vs_estimate": 12.0,
    "actual_delivery_days": 24.0,
    "carrier_delay_days": 18.0,
    "last_mile_days": 5.0,
    "estimated_delivery_days": 12.0
})

scenario_complex = base_order.copy()
scenario_complex.update({
    "scenario": "C - high value, multi-seller, multi-item",
    "is_late": 0,
    "seller_count": 3,
    "item_count": 4,
    "total_price": 650.0,
    "avg_price": 162.5,
    "total_freight": 95.0,
    "avg_freight": 23.75,
    "freight_to_price_ratio": 95.0 / 650.0,
    "total_payment_value": 745.0,
    "main_payment_type": "credit_card",
    "max_payment_installments": 10.0
})

scenario_far_late = base_order.copy()
scenario_far_late.update({
    "scenario": "D - cross-state late order",
    "is_late": 1,
    "delivery_delay_vs_estimate": 15.0,
    "actual_delivery_days": 32.0,
    "estimated_delivery_days": 17.0,
    "carrier_delay_days": 22.0,
    "last_mile_days": 8.0,
    "customer_state": "RJ",
    "seller_state": "SP",
    "same_customer_seller_state": 0,
    "main_product_category": "health_beauty"
})

scenario_df = pd.DataFrame([
    scenario_on_time,
    scenario_late,
    scenario_complex,
    scenario_far_late,
])

scenario_predictions = predict_batch(scenario_df, model, preprocessor, schema)
scenario_predictions[[
    "scenario",
    "low_rating_risk",
    "low_rating_risk_percentage",
    "risk_level",
    "is_late",
    "delivery_delay_vs_estimate",
    "actual_delivery_days",
    "seller_count",
    "item_count",
    "total_payment_value",
    "customer_state",
    "seller_state",
    "main_product_category"
]]

,scenario,low_rating_risk,low_rating_risk_percentage,risk_level,is_late,delivery_delay_vs_estimate,actual_delivery_days,seller_count,item_count,total_payment_value,customer_state,seller_state,main_product_category
0,"A - on-time, local, simple order",0.0610,6.10,Low,0,-7.000000,8.000000,1,1,38.71,SP,SP,housewares
1,"B - late delivery, long carrier delay",0.7832,78.32,High,1,12.000000,24.000000,1,1,38.71,SP,SP,housewares
2,"C - high value, multi-seller, multi-item",0.2500,25.00,Low,0,-7.107488,8.436574,3,4,745.00,SP,SP,housewares
3,D - cross-state late order,0.7832,78.32,High,1,15.000000,32.000000,1,1,38.71,RJ,SP,health_beauty


## 6. Input Manual Sendiri

Ubah nilai pada dictionary `my_order` di bawah untuk mencoba kasus lain. Pastikan semua fitur sesuai `schema['feature_columns']`.

In [8]:
my_order = {
    "approval_delay_days": 0.2,
    "carrier_delay_days": 12.0,
    "last_mile_days": 4.0,
    "estimated_delivery_days": 10.0,
    "actual_delivery_days": 18.0,
    "delivery_delay_vs_estimate": 8.0,
    "is_late": 1,
    "item_count": 2,
    "seller_count": 1,
    "total_price": 180.0,
    "avg_price": 90.0,
    "total_freight": 35.0,
    "avg_freight": 17.5,
    "freight_to_price_ratio": 35.0 / 180.0,
    "total_payment_value": 215.0,
    "payment_method_count": 1.0,
    "main_payment_type": "credit_card",
    "max_payment_installments": 4.0,
    "customer_state": "RJ",
    "seller_state": "SP",
    "same_customer_seller_state": 0,
    "main_product_category": "bed_bath_table",
    "product_weight_g": 1200.0,
    "product_volume_cm3": 12000.0,
}

predict_risk(my_order, model, preprocessor, schema)

{'low_rating_risk': 0.7832,
 'low_rating_risk_percentage': 78.32,
 'risk_level': 'High'}

## 7. Cara Membaca Output

- `low_rating_risk`: probabilitas model bahwa order masuk low rating (`review_score <= 2`).
- `low_rating_risk_percentage`: probabilitas yang sama dalam format persen.
- `risk_level`: label bisnis berbasis threshold:
  - `Low`: probability < 0.30
  - `Medium`: 0.30 <= probability < 0.50
  - `High`: probability >= 0.50

Gunakan notebook ini untuk demo simulasi skenario, bukan untuk menyimpulkan penyebab absolut low rating.